# 04. Comparaciones finales y verificación

Este cuaderno recalcula el análisis completo y contrasta sus 35 CSV con las referencias congeladas. Después muestra las comparaciones de calidad, su soporte y los descriptivos de generación desde la ejecución nueva.

Run All tarda más que los cuadernos de inspección porque repite las estimaciones bootstrap. No llama a modelos ni cambia los documentos.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

## Ejecutar el reanálisis completo

El programa común encadena validación, recuperación, explicitud, comparación de generadores y comparación de recuperadores dentro de cada generador. Utiliza un identificador nuevo en cada ejecución.

In [2]:
LOGS = new_run('comparacion_final')
run_id = 'notebook_final_' + LOGS.name.rsplit('_', 1)[-1]
report = json.loads(run_script('scripts/reproduce_offline.py', '--run-id', run_id, log_dir=LOGS))
RUN = ROOT / report['output']
assert report['compared_csv_count'] == 35
assert report['new_inference'] is False and report['documents_rendered'] is False
checks = [{'etapa': stage, 'archivo': name, 'identico': True}
          for stage, files in report['csv_sha256_equal_to_frozen'].items() for name in files]
assert len(checks) == 35
display(pd.DataFrame(checks))
print('Salidas:', RUN.relative_to(ROOT))

,etapa,archivo,identico
0,retrieval,aggregate_metrics.csv,True
1,retrieval,before_after.csv,True
2,retrieval,per_base.csv,True
3,retrieval,per_query.csv,True
4,paired,before_after_paired.csv,True
5,paired,between_method_effect_differences.csv,True
6,paired,paired_differences_by_base.csv,True
7,paired,paired_explicitness_v3.csv,True
8,paired,paired_grade2_eligible.csv,True
9,capacity,coverage.csv,True


Salidas: outputs/local_runs/final_experiment_20260930/REPLAY_notebook_final_c4a6eeadf67a


In [3]:
validation = json.loads((RUN / 'validation.json').read_text())
assert validation['table_count'] == 11 and validation['numeric_cells'] == 389
display(pd.DataFrame({'comprobacion': ['Tablas numericas', 'Valores numericos', 'CSV reproducidos'],
                      'n': [validation['table_count'], validation['numeric_cells'], report['compared_csv_count']]}))
display(pd.Series(validation['pdf'], name='valor').to_frame())

,comprobacion,n
0,Tablas numericas,11
1,Valores numericos,389
2,CSV reproducidos,35


,valor
main_pages,57
sponsor_pages,9
appendix_pages,9
main_table_captions,24


## Recuperadores dentro de cada generador

El análisis intersecta las formulaciones disponibles entre las cuatro estrategias para cada generador y criterio. Primero promedia dentro de pregunta base y después entre bases. Así evita que una estrategia parezca mejor por haber sido evaluada sobre preguntas distintas.

In [4]:
main = pd.read_csv(RUN / 'final/main_table.csv', na_values=['', 'NA', 'na'])
assert len(main) == 12
display(main[['generator_model', 'retrieval_condition', 'faithfulness_mean',
              'factual_correctness_mean', 'behavior_ok_mean', 'citations_ok_mean']])

,generator_model,retrieval_condition,faithfulness_mean,factual_correctness_mean,behavior_ok_mean,citations_ok_mean
0,gemma4:e4b,bm25,1.805556,1.500000,1.568182,0.954545
1,gemma4:e4b,dense,1.833333,1.583333,1.590909,0.931818
2,gemma4:e4b,rrf,1.750000,1.555556,1.590909,1.000000
3,gemma4:e4b,rerank,1.861111,1.472222,1.500000,0.931818
4,qwen3.5:4b,bm25,1.522727,1.590909,1.615385,0.903846
5,qwen3.5:4b,dense,1.772727,1.613636,1.576923,0.903846
6,qwen3.5:4b,rrf,1.772727,1.636364,1.711538,0.903846
7,qwen3.5:4b,rerank,1.931818,1.681818,1.692308,0.903846
8,qwen3.5:9b,bm25,1.886364,1.568182,1.673077,0.980769
9,qwen3.5:9b,dense,1.795455,1.613636,1.750000,0.961538


Las columnas de disponibilidad explican cuántas observaciones quedan fuera del soporte común. Las abstenciones se mantienen en los criterios de conducta y citas; las métricas de contenido se restringen a los casos previstos en el protocolo.

In [5]:
availability = pd.read_csv(RUN / 'final/availability.csv', na_values=['', 'NA', 'na'])
assert len(availability) == 48
display(availability[['generator_model', 'retrieval_condition', 'criterion',
                      'n_accepted', 'n_in_scope', 'n_unavailable', 'n_common', 'n_numeric_excluded']])

,generator_model,retrieval_condition,criterion,n_accepted,n_in_scope,n_unavailable,n_common,n_numeric_excluded
0,gemma4:e4b,bm25,faithfulness,48,40,0,34,6
1,gemma4:e4b,dense,faithfulness,49,41,0,34,7
2,gemma4:e4b,rrf,faithfulness,44,37,0,34,3
3,gemma4:e4b,rerank,faithfulness,47,39,0,34,5
4,gemma4:e4b,bm25,factual_correctness,48,40,0,33,7
5,gemma4:e4b,dense,factual_correctness,49,41,1,33,7
6,gemma4:e4b,rrf,factual_correctness,44,37,1,33,3
7,gemma4:e4b,rerank,factual_correctness,47,39,0,33,6
8,gemma4:e4b,bm25,behavior_ok,48,48,0,41,7
9,gemma4:e4b,dense,behavior_ok,49,49,0,41,8


Cada diferencia de la tabla siguiente es **estrategia de destino menos estrategia de origen**. Los intervalos bootstrap son puntuales, descriptivos y sin ajuste por la familia de comparaciones. Un intervalo que incluye cero no demuestra equivalencia.

In [6]:
contrasts = pd.read_csv(RUN / 'final/paired_contrasts.csv', na_values=['', 'NA', 'na'])
assert len(contrasts) == 72
display(contrasts[['generator_model', 'criterion', 'from_condition', 'to_condition',
                   'diff', 'ic95_lo', 'ic95_hi', 'n_cells', 'n_base']])

,generator_model,criterion,from_condition,to_condition,diff,ic95_lo,ic95_hi,n_cells,n_base
0,gemma4:e4b,faithfulness,bm25,dense,0.027778,-0.194444,0.277778,34,18
1,gemma4:e4b,faithfulness,bm25,rrf,-0.055556,-0.222222,0.083333,34,18
2,gemma4:e4b,faithfulness,bm25,rerank,0.055556,-0.194444,0.333333,34,18
3,gemma4:e4b,faithfulness,dense,rrf,-0.083333,-0.222222,0.055556,34,18
4,gemma4:e4b,faithfulness,dense,rerank,0.027778,-0.138889,0.194444,34,18
5,gemma4:e4b,faithfulness,rrf,rerank,0.111111,-0.083333,0.277778,34,18
6,gemma4:e4b,factual_correctness,bm25,dense,0.083333,-0.194444,0.361111,33,18
7,gemma4:e4b,factual_correctness,bm25,rrf,0.055556,-0.083333,0.222222,33,18
8,gemma4:e4b,factual_correctness,bm25,rerank,-0.027778,-0.277778,0.250000,33,18
9,gemma4:e4b,factual_correctness,dense,rrf,-0.027778,-0.250000,0.194444,33,18


## Comparación de generadores

La comparación entre generadores utiliza su propio soporte emparejado. La tabla permite examinar los criterios de contenido sin mezclar sus denominadores con los de la comparación anterior. El índice externo de capacidad no se utiliza aquí como una medición independiente de calidad.

In [7]:
generators = pd.read_csv(RUN / 'capacity/h6_niveles_common.csv', na_values=['', 'NA', 'na'])
display(generators.loc[generators['scope'] == 'primary_content',
    ['generator_model', 'faithfulness', 'faithfulness_n_base', 'faithfulness_n_cells',
     'factual_correctness', 'factual_correctness_n_base', 'factual_correctness_n_cells']])
display(pd.read_csv(RUN / 'capacity/h5_comparaciones_common.csv', na_values=['', 'NA', 'na']))

,generator_model,faithfulness,faithfulness_n_base,faithfulness_n_cells,factual_correctness,factual_correctness_n_base,factual_correctness_n_cells
0,gemma4:e4b,1.783929,22,157,1.498701,22,154
1,qwen3.5:4b,1.742965,22,157,1.598377,22,154
2,qwen3.5:9b,1.898647,22,157,1.649242,22,154


,scope,generator_model,criterion,from_condition,to_condition,diff,ic95_lo,ic95_hi,n_cells,n_base,positive,negative,ties,sign_n,sign_p_two_sided
0,primary_content,gemma4:e4b,faithfulness,bm25,dense,0.166667,-0.071429,0.452381,39,21,4,3,14,7,1.000000
1,primary_content,gemma4:e4b,faithfulness,bm25,rrf,0.000000,-0.175000,0.175000,36,20,3,3,14,6,1.000000
2,primary_content,gemma4:e4b,faithfulness,bm25,rerank,0.184211,-0.078947,0.500000,37,19,5,3,11,8,0.726562
3,primary_content,gemma4:e4b,faithfulness,dense,rrf,-0.075000,-0.200000,0.050000,36,20,1,3,16,4,0.625000
4,primary_content,gemma4:e4b,faithfulness,dense,rerank,0.125000,-0.100000,0.375000,37,20,3,1,16,4,0.625000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,sensitivity_abstain,qwen3.5:9b,citations_ok,bm25,rrf,0.000000,0.000000,0.000000,7,4,0,0,4,0,1.000000
212,sensitivity_abstain,qwen3.5:9b,citations_ok,bm25,rerank,0.000000,0.000000,0.000000,8,4,0,0,4,0,1.000000
213,sensitivity_abstain,qwen3.5:9b,citations_ok,dense,rrf,0.000000,0.000000,0.000000,7,4,0,0,4,0,1.000000
214,sensitivity_abstain,qwen3.5:9b,citations_ok,dense,rerank,0.000000,0.000000,0.000000,8,4,0,0,4,0,1.000000


## Tiempos y cobertura

Los tiempos proceden de la generación registrada e incluyen los intentos correctivos acumulados. No incluyen recuperación ni evaluación, y no constituyen un benchmark end-to-end controlado. Los fallos se muestran aparte de los tiempos medios de las respuestas aceptadas.

In [8]:
operations = pd.read_csv(RUN / 'final/operational.csv', na_values=['', 'NA', 'na'])
display(operations)
cells = pd.read_csv(RUN / 'final/operational_cells.csv', na_values=['', 'NA', 'na'])
assert len(cells) == 624 and cells['status'].value_counts().to_dict() == {'accepted': 604, 'failed': 20}
display(cells.groupby(['generator_model', 'status'])
        .agg(unidades=('qid', 'size'), intentos=('generation_attempts', 'sum'),
             segundos_mediana=('latency_s', 'median'), segundos_maximo=('latency_s', 'max')))

,generator_model,model,retrieval_condition,latency_s_mean,prompt_tokens_mean,completion_tokens_mean,accepted,failed,latency_s_n_numeric,prompt_tokens_n_numeric,completion_tokens_n_numeric
0,gemma4:e4b,Gemma,bm25,2.810247,2405.854167,126.291667,48,4,48,48,48
1,gemma4:e4b,Gemma,dense,3.135264,2491.102041,153.448980,49,3,49,49,49
2,gemma4:e4b,Gemma,rrf,2.985390,2436.386364,141.909091,44,8,44,44,44
3,gemma4:e4b,Gemma,rerank,2.973576,2415.042553,137.957447,47,5,47,47,47
4,qwen3.5:4b,Qwen4,bm25,2.731062,2391.576923,97.942308,52,0,52,52,52
5,qwen3.5:4b,Qwen4,dense,2.536664,2398.403846,86.923077,52,0,52,52,52
6,qwen3.5:4b,Qwen4,rrf,2.555797,2369.730769,94.884615,52,0,52,52,52
7,qwen3.5:4b,Qwen4,rerank,2.632446,2507.365385,98.269231,52,0,52,52,52
8,qwen3.5:9b,Qwen9,bm25,3.905620,2343.596154,88.134615,52,0,52,52,52
9,qwen3.5:9b,Qwen9,dense,3.866592,2306.153846,96.211538,52,0,52,52,52


unidades  intentos  segundos_mediana  \
generator_model status                                           
gemma4:e4b      accepted       188       191          2.851134   
                failed          20        60         12.920351   
qwen3.5:4b      accepted       208       215          2.597193   
qwen3.5:9b      accepted       208       208          3.865677   

                          segundos_maximo  
generator_model status                     
gemma4:e4b      accepted         9.054686  
                failed          13.314372  
qwen3.5:4b      accepted         5.638997  
qwen3.5:9b      accepted        12.810835

## Figuras de las memorias, opcionales

La opción siguiente ejecuta el código R literal de las fuentes y guarda diez SVG en otro destino nuevo. Requiere R con dplyr, ggplot2, jsonlite, knitr y tibble. Se deja desactivada para que el reanálisis numérico pueda hacerse solo con el entorno Python.

In [9]:
REGENERAR_FIGURAS = True
if REGENERAR_FIGURAS:
    figures = json.loads(run_script('scripts/reproduce_figures.py', '--run-id', run_id, log_dir=LOGS))
    assert figures['r_chunks_executed'] == 11 and figures['svg_count'] == 10
    display(pd.DataFrame({'figura': figures['svg_files']}))

,figura
0,fig-chunks-capitulo.svg
1,fig-longitud-chunks.svg
2,fig-recuperacion-cortes.svg
3,fig-recuperadores-contenido.svg
4,fig-diferencias-emparejadas.svg
5,fig-capacidad.svg
6,fig-tiempo-generacion.svg
7,fig-completitud-final.svg
8,fig-sponsor-recuperacion.svg
9,fig-sponsor-calidad.svg


La reproducción queda comprobada cuando los 35 CSV coinciden con las referencias y los validadores aceptan las tablas y los documentos. Eso acredita el reanálisis de las fuentes incluidas; no una nueva generación de respuestas ni una repetición de la revisión humana.